In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from datetime import datetime
import traceback
import warnings
import pickle
import joblib
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
import logging

warnings.filterwarnings('ignore')

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("triage_model_evaluation.log")
    ]
)
logger = logging.getLogger()

def train_and_evaluate_advanced_models(X_train_path, y_train_path, X_test_path, y_test_path, output_dir):
    """Entrena y evalúa modelos avanzados con optimización de hiperparámetros"""
    
    model_name = os.path.basename(X_train_path).replace("_X_train.parquet", "")
    print(f"\n{'='*50}\nProcesando: {model_name}\n{'='*50}")
    logger.info(f"Iniciando procesamiento para {model_name}")
    
    try:
        # Cargar datos
        print("Cargando datos...")
        X_train = pq.read_table(X_train_path).to_pandas()
        y_train = pq.read_table(y_train_path).to_pandas().squeeze()
        X_test = pq.read_table(X_test_path).to_pandas()
        y_test = pq.read_table(y_test_path).to_pandas().squeeze()
        
        # Información sobre el dataset
        print(f"Datos de entrenamiento: {X_train.shape}")
        print(f"Datos de prueba: {X_test.shape}")
        print(f"Distribución de clases (train): {y_train.value_counts().to_dict()}")
        
        # Verificar si hay NaN en los datos
        nan_count_train = X_train.isna().sum().sum()
        nan_count_test = X_test.isna().sum().sum()
        
        if nan_count_train > 0 or nan_count_test > 0:
            print(f"¡Advertencia! Valores NaN detectados: Train={nan_count_train}, Test={nan_count_test}")
            print("Reemplazando con la mediana...")
            
            # Calcular medianas para imputación
            median_values = X_train.median()
            X_train = X_train.fillna(median_values)
            X_test = X_test.fillna(median_values)
        
        # Selección de características
        print("Seleccionando características...")
        selector = SelectKBest(f_classif, k=min(100, X_train.shape[1]))
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)
        
        # Obtener índices de características seleccionadas
        selected_features = selector.get_support(indices=True)
        feature_names = X_train.columns[selected_features].tolist()
        
        print(f"Características seleccionadas: {len(feature_names)}")
        
        # Definir modelos a evaluar
        models = {
            'RandomForest': RandomForestClassifier(random_state=42, class_weight='balanced'),
            'GradientBoosting': GradientBoostingClassifier(random_state=42),
            'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
        }
        
        # CORREGIDO: Definir parámetros para Grid Search correctamente para Pipeline
        param_grids = {
            'RandomForest': {
                'model__n_estimators': [100, 200],
                'model__max_depth': [None, 10, 20],
                'model__min_samples_split': [2, 5],
                'model__min_samples_leaf': [1, 2],
                'model__max_features': ['sqrt', 'log2']
            },
            'GradientBoosting': {
                'model__n_estimators': [100, 200],
                'model__learning_rate': [0.01, 0.1],
                'model__max_depth': [3, 5],
                'model__min_samples_split': [2, 5]
            },
            'LogisticRegression': {
                'model__C': [0.1, 1.0, 10.0],
                'model__penalty': ['l1', 'l2'],
                'model__solver': ['saga']
            }
        }
        
        # Lista para almacenar resultados
        results = []
        best_models = {}
        
        # Entrenar y evaluar cada modelo
        for model_type, model in models.items():
            print(f"\nEntrenando {model_type}...")
            
            # Definir pipeline
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model', model)
            ])
            
            # Grid Search
            grid_search = GridSearchCV(
                pipeline,
                param_grids[model_type],
                cv=5,
                scoring='f1_macro',
                n_jobs=-1,
                verbose=1
            )
            
            # Entrenamiento
            grid_search.fit(X_train_selected, y_train)
            
            # ... resto de tu código ...
            
            # Mejores parámetros
            best_params = grid_search.best_params_
            print(f"Mejores parámetros para {model_type}: {best_params}")
            
            # Mejor modelo
            best_model = grid_search.best_estimator_
            best_models[model_type] = best_model
            
            # Predicciones
            y_pred = best_model.predict(X_test_selected)
            y_proba = best_model.predict_proba(X_test_selected)
            
            # Métricas
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='macro')
            recall = recall_score(y_test, y_pred, average='macro')
            f1 = f1_score(y_test, y_pred, average='macro')
            
            # Calcular AUC para cada clase
            auc_scores = {}
            for i in sorted(np.unique(y_test)):
                if i-1 < y_proba.shape[1]:  # Verificar que el índice existe
                    try:
                        fpr, tpr, _ = roc_curve((y_test == i).astype(int), y_proba[:, i-1])
                        auc_value = auc(fpr, tpr)
                        auc_scores[f"Clase_{i}"] = auc_value
                    except Exception as e:
                        print(f"Error calculando AUC para clase {i}: {str(e)}")
                        auc_scores[f"Clase_{i}"] = np.nan
            
            # AUC promedio
            mean_auc = np.mean([v for v in auc_scores.values() if not np.isnan(v)])
            
            # Guardar resultados
            results.append({
                'model_name': f"{model_name}_{model_type}",
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'mean_auc': mean_auc,
                'auc_scores': auc_scores,
                'best_params': best_params
            })
            
            # Generar gráficos de ROC
            plt.figure(figsize=(10, 8))
            for i, (class_name, auc_value) in enumerate(auc_scores.items()):
                if not np.isnan(auc_value):
                    class_id = int(class_name.split('_')[1])
                    fpr, tpr, _ = roc_curve((y_test == class_id).astype(int), y_proba[:, class_id-1])
                    plt.plot(fpr, tpr, lw=2, label=f'{class_name} (AUC = {auc_value:.2f})')
            
            plt.plot([0, 1], [0, 1], 'k--', lw=2)
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('Tasa de Falsos Positivos')
            plt.ylabel('Tasa de Verdaderos Positivos')
            plt.title(f'Curva ROC - {model_type}')
            plt.legend(loc="lower right")
            
            # Guardar gráficos
            os.makedirs(os.path.join(output_dir, "plots"), exist_ok=True)
            plt.savefig(os.path.join(output_dir, "plots", f"{model_name}_{model_type}_roc.png"))
            plt.close()
            
            # Matriz de confusión
            plt.figure(figsize=(10, 8))
            cm = confusion_matrix(y_test, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                        xticklabels=sorted(np.unique(y_test)),
                        yticklabels=sorted(np.unique(y_test)))
            plt.title(f'Matriz de Confusión - {model_type}')
            plt.xlabel('Predicción')
            plt.ylabel('Valor Real')
            plt.savefig(os.path.join(output_dir, "plots", f"{model_name}_{model_type}_cm.png"))
            plt.close()
            
            # Guardar modelo
            os.makedirs(os.path.join(output_dir, "models"), exist_ok=True)
            joblib.dump(best_model, os.path.join(output_dir, "models", f"{model_name}_{model_type}.joblib"))
            
            print(f"✅ Modelo {model_type} completado:\n"
                  f"   - Accuracy: {accuracy:.4f}\n"
                  f"   - F1 Score: {f1:.4f}\n"
                  f"   - AUC Promedio: {mean_auc:.4f}")
        
        # Crear y entrenar un modelo de ensamble (Voting)
        print("\nCreando modelo de ensamble (Voting)...")
        
        # Configurar el ensamble
        estimators = [(name, model) for name, model in best_models.items()]
        ensemble = VotingClassifier(estimators=estimators, voting='soft')
        
        # Entrenar el ensamble
        ensemble.fit(X_train_selected, y_train)
        
        # Evaluar el ensamble
        y_pred_ensemble = ensemble.predict(X_test_selected)
        y_proba_ensemble = ensemble.predict_proba(X_test_selected)
        
        # Métricas del ensamble
        accuracy_ensemble = accuracy_score(y_test, y_pred_ensemble)
        precision_ensemble = precision_score(y_test, y_pred_ensemble, average='macro')
        recall_ensemble = recall_score(y_test, y_pred_ensemble, average='macro')
        f1_ensemble = f1_score(y_test, y_pred_ensemble, average='macro')
        
        # Calcular AUC para cada clase (ensamble)
        auc_scores_ensemble = {}
        for i in sorted(np.unique(y_test)):
            if i-1 < y_proba_ensemble.shape[1]:
                try:
                    fpr, tpr, _ = roc_curve((y_test == i).astype(int), y_proba_ensemble[:, i-1])
                    auc_value = auc(fpr, tpr)
                    auc_scores_ensemble[f"Clase_{i}"] = auc_value
                except Exception as e:
                    print(f"Error calculando AUC para clase {i} (ensamble): {str(e)}")
                    auc_scores_ensemble[f"Clase_{i}"] = np.nan
        
        # AUC promedio (ensamble)
        mean_auc_ensemble = np.mean([v for v in auc_scores_ensemble.values() if not np.isnan(v)])
        
        # Agregar resultados del ensamble
        results.append({
            'model_name': f"{model_name}_Ensemble",
            'accuracy': accuracy_ensemble,
            'precision': precision_ensemble,
            'recall': recall_ensemble,
            'f1': f1_ensemble,
            'mean_auc': mean_auc_ensemble,
            'auc_scores': auc_scores_ensemble,
            'best_params': 'N/A (Ensemble)'
        })
        
        # Guardar ensamble
        joblib.dump(ensemble, os.path.join(output_dir, "models", f"{model_name}_Ensemble.joblib"))
        
        # Matriz de confusión del ensamble
        plt.figure(figsize=(10, 8))
        cm_ensemble = confusion_matrix(y_test, y_pred_ensemble)
        sns.heatmap(cm_ensemble, annot=True, fmt='d', cmap='Blues',
                    xticklabels=sorted(np.unique(y_test)),
                    yticklabels=sorted(np.unique(y_test)))
        plt.title('Matriz de Confusión - Ensemble')
        plt.xlabel('Predicción')
        plt.ylabel('Valor Real')
        plt.savefig(os.path.join(output_dir, "plots", f"{model_name}_Ensemble_cm.png"))
        plt.close()
        
        # ROC del ensamble
        plt.figure(figsize=(10, 8))
        for i, (class_name, auc_value) in enumerate(auc_scores_ensemble.items()):
            if not np.isnan(auc_value):
                class_id = int(class_name.split('_')[1])
                fpr, tpr, _ = roc_curve((y_test == class_id).astype(int), y_proba_ensemble[:, class_id-1])
                plt.plot(fpr, tpr, lw=2, label=f'{class_name} (AUC = {auc_value:.2f})')
        
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('Tasa de Falsos Positivos')
        plt.ylabel('Tasa de Verdaderos Positivos')
        plt.title('Curva ROC - Ensemble')
        plt.legend(loc="lower right")
        plt.savefig(os.path.join(output_dir, "plots", f"{model_name}_Ensemble_roc.png"))
        plt.close()
        
        print(f"✅ Modelo Ensemble completado:\n"
              f"   - Accuracy: {accuracy_ensemble:.4f}\n"
              f"   - F1 Score: {f1_ensemble:.4f}\n"
              f"   - AUC Promedio: {mean_auc_ensemble:.4f}")
        
        # Guardar resultados en CSV
        results_df = pd.DataFrame(results)
        results_df.to_csv(os.path.join(output_dir, f"{model_name}_results.csv"), index=False)
        
        # Guardar información sobre las características seleccionadas
        feature_importance_df = pd.DataFrame({
            'feature': feature_names,
            'index': selected_features
        })
        feature_importance_df.to_csv(os.path.join(output_dir, f"{model_name}_selected_features.csv"), index=False)
        
        # Determinar el mejor modelo basado en F1
        best_idx = results_df['f1'].idxmax()
        best_model_info = results_df.iloc[best_idx]
        
        print(f"\n🏆 Mejor modelo para {model_name}: {best_model_info['model_name']}")
        print(f"   - F1 Score: {best_model_info['f1']:.4f}")
        print(f"   - Accuracy: {best_model_info['accuracy']:.4f}")
        print(f"   - AUC Promedio: {best_model_info['mean_auc']:.4f}")
        
        return results
    
    except Exception as e:
        print(f"\n❌ Error procesando {model_name}: {str(e)}")
        logger.error(f"Error procesando {model_name}: {str(e)}")
        traceback.print_exc()
        return None

def generate_pdf_report(all_results, output_dir):
    """Genera un informe PDF con los resultados de todos los modelos"""
    
    if not all_results:
        print("No hay resultados para generar un informe.")
        return
    
    # Aplanar resultados si es necesario
    flat_results = []
    for results_list in all_results:
        if isinstance(results_list, list):
            flat_results.extend(results_list)
        else:
            flat_results.append(results_list)
    
    # Convertir a DataFrame
    results_df = pd.DataFrame(flat_results)
    
    # Ordenar por F1 score
    results_df = results_df.sort_values('f1', ascending=False).reset_index(drop=True)
    
    # Preparar PDF
    pdf_path = os.path.join(output_dir, f"informe_modelos_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf")
    doc = SimpleDocTemplate(pdf_path, pagesize=letter)
    elements = []
    
    # Estilos
    styles = getSampleStyleSheet()
    title_style = styles['Heading1']
    subtitle_style = styles['Heading2']
    normal_style = styles['Normal']
    
    # Título
    elements.append(Paragraph("Informe de Evaluación de Modelos de Triage", title_style))
    elements.append(Spacer(1, 12))
    
    # Fecha
    elements.append(Paragraph(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}", normal_style))
    elements.append(Spacer(1, 12))
    
    # Resumen
    elements.append(Paragraph("Resumen de Resultados", subtitle_style))
    elements.append(Spacer(1, 6))
    
    # Tabla de resultados (Top 10)
    top_models = results_df.head(10)
    table_data = [["Modelo", "F1 Score", "Accuracy", "Precision", "Recall", "AUC"]]
    
    for _, row in top_models.iterrows():
        table_data.append([
            row['model_name'],
            f"{row['f1']:.4f}",
            f"{row['accuracy']:.4f}",
            f"{row['precision']:.4f}",
            f"{row['recall']:.4f}",
            f"{row['mean_auc']:.4f}"
        ])
    
    # Crear tabla
    table = Table(table_data, colWidths=[180, 70, 70, 70, 70, 70])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('ALIGN', (1, 1), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ]))
    
    elements.append(table)
    elements.append(Spacer(1, 12))
    
    # Mejor modelo
    best_model = results_df.iloc[0]
    elements.append(Paragraph("Mejor Modelo", subtitle_style))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph(f"Nombre: {best_model['model_name']}", normal_style))
    elements.append(Paragraph(f"F1 Score: {best_model['f1']:.4f}", normal_style))
    elements.append(Paragraph(f"Accuracy: {best_model['accuracy']:.4f}", normal_style))
    elements.append(Paragraph(f"AUC Promedio: {best_model['mean_auc']:.4f}", normal_style))
    
    if isinstance(best_model['best_params'], dict):
        elements.append(Paragraph("Mejores Parámetros:", normal_style))
        for param, value in best_model['best_params'].items():
            elements.append(Paragraph(f"   - {param}: {value}", normal_style))
    else:
        elements.append(Paragraph(f"Parámetros: {best_model['best_params']}", normal_style))
    
    elements.append(Spacer(1, 12))
    
    # Conclusiones
    elements.append(Paragraph("Conclusiones", subtitle_style))
    elements.append(Spacer(1, 6))
    
    # Comparación de tipos de modelos
    model_types = results_df['model_name'].apply(lambda x: x.split('_')[-1])
    avg_f1_by_type = results_df.groupby(model_types)['f1'].mean().sort_values(ascending=False)
    
    elements.append(Paragraph("Rendimiento promedio por tipo de modelo:", normal_style))
    for model_type, avg_f1 in avg_f1_by_type.items():
        elements.append(Paragraph(f"   - {model_type}: F1 = {avg_f1:.4f}", normal_style))
    
    elements.append(Spacer(1, 12))
    
    # Recomendaciones
    elements.append(Paragraph("Recomendaciones", subtitle_style))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph("1. Utilizar el modelo con mejor F1 Score para un balance entre precisión y recall.", normal_style))
    elements.append(Paragraph("2. Considerar el modelo Ensemble para obtener predicciones más robustas.", normal_style))
    elements.append(Paragraph("3. Para casos críticos donde se requiere alta sensibilidad, priorizar modelos con mayor recall.", normal_style))
    
    # Generar PDF
    doc.build(elements)
    print(f"Informe generado: {pdf_path}")
    return pdf_path

def main():
    """Función principal de ejecución"""
    
    # Directorio de entrada con los datos de entrenamiento y prueba
    input_root = r"C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments\train_test_splits_20250304_233052"
    output_dir = os.path.join(input_root, "advanced_models")
    os.makedirs(output_dir, exist_ok=True)
    
    # Directorios para organizar resultados
    os.makedirs(os.path.join(output_dir, "models"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "plots"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "logs"), exist_ok=True)
    
    # Contador de procesamiento
    processed_count = 0
    error_count = 0
    all_results = []
    
    # Buscar los conjuntos de datos de Model-Based que han tenido los mejores resultados según los notebooks
    target_models = [
        "MaxAbs_Model-Based_selected",
        "MinMax_Model-Based_selected",
        "Robust_Model-Based_selected",
        "Standard_Model-Based_selected"
    ]
    
    # Buscar archivos de forma organizada
    for root, _, files in os.walk(input_root):
        # Mapeo de prefijos a archivos
        prefix_files = {}
        for file in files:
            if file.endswith(".parquet"):
                # Extraer información del archivo
                if "_X_train" in file:
                    prefix = file.replace("_X_train.parquet", "")
                    if prefix not in prefix_files:
                        prefix_files[prefix] = {}
                    prefix_files[prefix]['X_train'] = os.path.join(root, file)
                elif "_y_train" in file:
                    prefix = file.replace("_y_train.parquet", "")
                    if prefix not in prefix_files:
                        prefix_files[prefix] = {}
                    prefix_files[prefix]['y_train'] = os.path.join(root, file)
                elif "_X_test" in file:
                    prefix = file.replace("_X_test.parquet", "")
                    if prefix not in prefix_files:
                        prefix_files[prefix] = {}
                    prefix_files[prefix]['X_test'] = os.path.join(root, file)
                elif "_y_test" in file:
                    prefix = file.replace("_y_test.parquet", "")
                    if prefix not in prefix_files:
                        prefix_files[prefix] = {}
                    prefix_files[prefix]['y_test'] = os.path.join(root, file)
        
        # Procesar los conjuntos completos que coincidan con los modelos objetivo
        for prefix, files_dict in prefix_files.items():
            # Verificar si es uno de los modelos objetivo
            matched_model = None
            for target in target_models:
                if target in prefix:
                    matched_model = target
                    break
            
            if matched_model and len(files_dict) == 4:  # Todos los archivos presentes
                logger.info(f"Procesando modelo objetivo: {prefix}")
                
                try:
                    results = train_and_evaluate_advanced_models(
                        files_dict['X_train'],
                        files_dict['y_train'],
                        files_dict['X_test'],
                        files_dict['y_test'],
                        output_dir
                    )
                    
                    if results:
                        all_results.append(results)
                        processed_count += 1
                        logger.info(f"✅ {prefix} procesado correctamente")
                except Exception as e:
                    error_count += 1
                    logger.error(f"❌ Error inesperado procesando {prefix}: {str(e)}")
                    traceback.print_exc()
    
    # Generar informe final
    if all_results:
        # Guardar todos los resultados
        flat_results = []
        for results_list in all_results:
            if isinstance(results_list, list):
                flat_results.extend(results_list)
            else:
                flat_results.append(results_list)
        
        combined_results_df = pd.DataFrame(flat_results)
        combined_results_df.to_csv(os.path.join(output_dir, "all_models_results.csv"), index=False)
        
        # Generar reporte PDF
        generate_pdf_report(all_results, output_dir)
        
        # Mostrar resumen
        best_model_idx = combined_results_df['f1'].idxmax()
        best_model = combined_results_df.iloc[best_model_idx]
        
        print(f"\n🏆 MEJOR MODELO GENERAL 🏆")
        print(f"Nombre: {best_model['model_name']}")
        print(f"F1 Score: {best_model['f1']:.4f}")
        print(f"Accuracy: {best_model['accuracy']:.4f}")
        print(f"AUC Promedio: {best_model['mean_auc']:.4f}")
    
    # Resumen final
    print(f"\nResumen de ejecución:")
    print(f"- Conjuntos de datos procesados: {processed_count}")
    print(f"- Errores: {error_count}")
    print(f"- Modelos evaluados: {len(flat_results) if 'flat_results' in locals() else 0}")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Error en la ejecución principal: {str(e)}")
        traceback.print_exc()


Procesando: MaxAbs_Model-Based_selected
Cargando datos...
Datos de entrenamiento: (621805, 80)
Datos de prueba: (112098, 80)
Distribución de clases (train): {2: 124361, 3: 124361, 5: 124361, 4: 124361, 1: 124361}
Seleccionando características...
Características seleccionadas: 80

Entrenando RandomForest...
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Mejores parámetros para RandomForest: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}
✅ Modelo RandomForest completado:
   - Accuracy: 1.0000
   - F1 Score: 1.0000
   - AUC Promedio: 1.0000

Entrenando GradientBoosting...
Fitting 5 folds for each of 16 candidates, totalling 80 fits


KeyboardInterrupt: 